In [24]:
from functools import reduce
from datetime import datetime
from pyspark.sql.functions import col
from pyspark.sql.functions import lit

StatementMeta(, a06492f8-719f-463b-b62f-4a60739b3aa2, 26, Finished, Available, Finished, False)

In [17]:
now=str(datetime.utcnow())
now

StatementMeta(, a06492f8-719f-463b-b62f-4a60739b3aa2, 19, Finished, Available, Finished, False)

'2026-04-29 08:03:29.166759'

In [18]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
df_inserted=spark.sql("""
    SELECT
    a.NId,
    a.Id,
    a.Status,
    a.ParentOrder,
    a.Process,
    a.AsPlanned,
    a.ActualEndTime,
    a.ActualStartTime,
    a.CreationDate,
    a.DueDate,
    a.Enterprise,
    a.ERPOrder,
    a.ParentBatch,
    a.EstimatedEndTime,
    a.EstimatedStartTime,
    a.InitialQuantity,
    a.IsUnderScheduling,
    a.WorkOrderName,
    a.Notes,
    a.PBOPIdentID,
    a.Plant,
    a.Priority,
    a.Sequence,
    a.ProcessNId,
    a.ProcessRevision,
    a.ProcessUId,
    a.ProducedQuantity,
    a.ReworkedQuantity,
    a.SchedulingDate,
    a.ScrappedQuantity,
    a.PoC,
    a.ReworkOfOrder,
    a.WorkOrderOperationId,
    a.FinalMaterial,
    a.FinalMaterialName,
    a.FinalMaterialRevision,
    a.ProductionType,
    a.Machine,
    a.LineaRepartoNId,
    a.LineaReparto,
    a.ActualUsedMachineName,
    a.ToBeUsedMachineName,
    a.LineaMontaggio,
    a.Company,
    a.OrderType,
    a.Resource,
    a.WashingMachineCode,
    a.LineaMontaggioNId,
    a.RowUpdated
    FROM staging.production_orders as a
    LEFT OUTER JOIN bronze.production_orders b
    on a.NId=b.NId
    WHERE b.NId is Null
""")

StatementMeta(, a06492f8-719f-463b-b62f-4a60739b3aa2, 20, Finished, Available, Finished, False)

In [20]:
df_inserted=df_inserted.withColumn("valid_from",lit(now))
df_inserted=df_inserted.withColumn("valid_from",col("valid_from").cast("timestamp"))
df_inserted=df_inserted.withColumn("valid_to",lit(None))
df_inserted=df_inserted.withColumn("valid_to",col("valid_to").cast("timestamp"))

StatementMeta(, a06492f8-719f-463b-b62f-4a60739b3aa2, 22, Finished, Available, Finished, False)

In [22]:
df_staging=spark.table("staging.production_orders")
df_bronze=spark.table("bronze.production_orders")

StatementMeta(, a06492f8-719f-463b-b62f-4a60739b3aa2, 24, Finished, Available, Finished, False)

In [29]:
key = "NId"

cols_to_check = [c for c in df_staging.columns if c != key]

joined = df_staging.alias("o").join(df_bronze.alias("n"), key, "inner")

condition = reduce(
    lambda a, b: a & b,
    [col(f"o.{c}").eqNullSafe(col(f"n.{c}")) for c in cols_to_check]
)

df_updated = joined.filter(~condition).select("o.*")

StatementMeta(, a06492f8-719f-463b-b62f-4a60739b3aa2, 31, Finished, Available, Finished, False)

In [30]:
df_updated=df_updated.withColumn("valid_from",lit(now))
df_updated=df_updated.withColumn("valid_from",col("valid_from").cast("timestamp"))
df_updated=df_updated.withColumn("valid_to",lit(None))
df_updated=df_updated.withColumn("valid_to",col("valid_to").cast("timestamp"))

StatementMeta(, a06492f8-719f-463b-b62f-4a60739b3aa2, 32, Finished, Available, Finished, False)

In [37]:
df_updated.createOrReplaceTempView("v_df_updated")

StatementMeta(, a06492f8-719f-463b-b62f-4a60739b3aa2, 39, Finished, Available, Finished, False)

In [40]:
spark.sql(f"""
MERGE INTO bronze.production_orders t
USING v_df_updated s
ON t.Nid = s.Nid
WHEN MATCHED THEN
  UPDATE SET t.valid_to = '{now}'
""")

StatementMeta(, a06492f8-719f-463b-b62f-4a60739b3aa2, 42, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [41]:
df_inserted.write.mode("append").saveAsTable("bronze.production_orders")

StatementMeta(, a06492f8-719f-463b-b62f-4a60739b3aa2, 43, Finished, Available, Finished, False)

In [42]:
df_updated.write.mode("append").saveAsTable("bronze.production_orders")

StatementMeta(, a06492f8-719f-463b-b62f-4a60739b3aa2, 44, Finished, Available, Finished, False)